# Metax (MACA) environment check

Run this inside a Metax DevEnvironment to confirm what the image carries and that the
GPU can actually be used by the account the environment runs as.

The device count reports the vendor's configured visible-device set. It can be correct
while every open of the device fails, so it is printed rather than asserted — the last
cell is the one that proves access, and it is also the first cell to touch a device.

Give it memory. That first touch initializes the MACA driver in this process, which adds
about 7.4 GiB of host memory — the run this was written from peaked near 8 GiB. An
environment of much under 16 GiB is therefore OOM-killed part-way through, at the last
cell, with everything above it already printed. That is the environment's memory limit
rather than a broken GPU.


In [ ]:
# The interpreter this kernel runs on. In the MACA images it is the vendor's
# /opt/conda python, which is the one that carries torch.
from platform import python_version
print(f"Python Version: {python_version()}")


In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
# The CUDA-compatibility level this MACA build is a port of.
print(f"CUDA-Compatible Version: {torch.version.cuda}")


In [ ]:
# The vendor's own stack version.
import pymxsml
pymxsml.mxSmlInit()
print(f"MetaX MACA Version: {pymxsml.mxSmlGetMacaVersion()}")


In [ ]:
# What the environment was configured to see. Reading it opens no device, which is
# why it can be right while access fails — see the last cell.
count = torch.cuda.device_count()
print(f"Total Physical GPUs: {count}")


In [ ]:
# The access check. `is_available()` needs the device to be openable by this account,
# which is what the cell above cannot show: a non-root environment can report the right
# count and then fail here. If it is False, the node's container runtime is not handing
# this account the device.
print(f"CUDA Available: {torch.cuda.is_available()}")

# The names, and a real operation. The line above already initialized the driver, so
# neither is the first touch; they follow it rather than sitting with the count, because
# a device that cannot be opened would otherwise raise before the access result printed.
print([torch.cuda.get_device_name(i) for i in range(count)])

x = torch.randn(256, 256, device="cuda")
print(f"256x256 matmul ok, sum = {float((x @ x).sum())}")
